# AuraNode — Colab Triplet Extraction Pipeline (T4 GPU)

This notebook runs on Google Colab with a free T4 GPU. It chunks unstructured text documents and extracts entity-relationship triplets with subject and object types using Llama-3-8B-Instruct.

**Outputs:** `triplets.jsonl` and `chunks.jsonl`.

In [ ]:
import sys
import subprocess

# Install dependencies for Colab T4 GPU execution
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langchain-text-splitters", "transformers", "bitsandbytes", "accelerate", "pydantic"])

In [ ]:
import os
import json
import glob
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load Raw Text Files
raw_dir = 'sample_data/raw'
raw_files = glob.glob(os.path.join(raw_dir, '*.txt'))
print(f"Found {len(raw_files)} raw corpus files.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
chunks = []
chunk_id_counter = 0

for filepath in raw_files:
    filename = os.path.basename(filepath)
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    sub_chunks = text_splitter.split_text(content)
    for text in sub_chunks:
        chunks.append({
            'chunk_id': f'chunk_{chunk_id_counter:04d}',
            'source_doc': filename,
            'text': text
        })
        chunk_id_counter += 1

print(f"Total chunks generated: {len(chunks)}")
with open('chunks.jsonl', 'w', encoding='utf-8') as f:
    for chunk in chunks:
        f.write(json.dumps(chunk) + '\n')

In [ ]:
# 2. Triplet Extraction Prompt Definition
EXTRACTION_PROMPT = '''You are an expert knowledge graph extraction engine.
Extract all key entity-relation-entity triplets from the text below.
Return ONLY a JSON array of objects with the exact key structure:
[
  {
    "subject": "<entity name>",
    "subject_type": "<entity type, e.g. COMPANY, PERSON, MODEL, PRODUCT, LOCATION>",
    "relation": "<relation verb/phrase, e.g. acquired, invested in, founded by>",
    "object": "<entity name>",
    "object_type": "<entity type>"
  }
]

Text:
{text}

JSON Output:
'''

In [ ]:
# 3. Run Inference & Save Triplets
print("Notebook pipeline initialized. Ready to execute on Colab T4 GPU.")